# 第2回 演習（用語集版）: LoRA で回答形式を学習し、学習前後を比較する

`lora_format_tuning.ipynb` と **同じ設定**（モデル、revision、LoRA の設定、エポック数、生成設定、採点関数）のまま、学習データと評価データだけを講座の用語集に差し替えたものです。

- 学習データ: `glossary_sft.jsonl`（58 語 × 言い換え 2 件 = 116 件）。assistant の説明文を `answer` に入れた `{"answer": ..., "needs_review": false}` を望ましい出力にします。
- 評価データ: 12 問（根拠あり 8、根拠なし 4）。根拠ありの 8 問は **講座固有の事実**（既定モデル名、環境変数名、ブランチ名、ファイルの場所、Pydantic AI の引数名など）を聞くもので、事前学習の一般知識では答えられず、用語集を学習して初めて答えられます。根拠なしの 4 問は用語集に無い語や講座運営の事実で、`needs_review: true` が正解です。

**元の演習との違いは次の 2 点だけです**

1. 元の演習は FAQ（5 項目）をプロンプトに渡していましたが、用語集は 58 項目あり `max_seq_len: 512` に収まらないので、プロンプトには渡しません。用語集の内容は学習で重みに入れます。
2. 元の演習は train に「根拠なし」の例が 4 件ありましたが、`glossary_sft.jsonl` には根拠なしの例がありません。「記載が無ければ `needs_review: true`」という振る舞いはシステムプロンプトの指示だけが頼りです。

評価は元と同じく **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。セクションの構成、セルの読み方（やること・見るところ・うまくいかないとき）も元のノートブックと同じです。

**元と変えた唯一の設定**: `max_new_tokens` を 96 から 192 にしています（用語集の説明文が JSON 込みで 96 トークンを超え、学習後の出力が途中で切れるため）。

**A100 向けの設定**: このノートブックは Colab の A100（40GB）で動かす前提です。元の T4 向け設定から変えたのは、計算精度を bf16 の混合精度にした点（`USE_AMP = True`。A100 は bf16 に対応し、T4 で問題になる `nan` の心配がない）と、セクション 8 の 4bit 計算精度を bf16 にした点だけで、バッチ 4 × 勾配累積 2、LoRA の設定、学習率、エポック数は元と同じです。重みは 32bit のまま保持します。

**所要時間の目安（A100）**: 学習データが 14 件から 116 件に増えたので、同じ 20 エポックでは 300 ステップ（元は 40 ステップ）になりますが、A100 なら学習は 5〜10 分です。T4 で動かす場合は `USE_AMP = False` に戻してください（20〜30 分）。

**使うもの**: Google Colab（A100 GPU）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。

版: 1.0（2026-10-31、元ノートブック 1.1 に基づく）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `NVIDIA A100-SXM4-40GB` のように表示されること（T4 でも動きますが、学習に 20〜30 分かかります）。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「A100 GPU」を選び、セルを実行し直します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → A100 GPU を選んでください。")
if "A100" not in gpu_info: print("A100 以外の GPU です。T4 の場合はセクション 4 の USE_AMP を False にしてください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB

A100 以外の GPU です。T4 の場合はセクション 4 の USE_AMP を False にしてください。


In [2]:
# Colab（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数などを 1 つの辞書 `CONFIG` にまとめます。値は元のノートブックと同じです（`work_dir` と `train_file` だけ用語集版の名前にしています）。

- `num_train_epochs: 20` は元と同じですが、学習データが 116 件なので 1 エポックは 15 ステップ（8 件 × 14 と端数 4 件）、20 エポックで 300 ステップです。A100 で 5〜10 分です。
- `max_new_tokens: 192`。**ここだけ元（96）と変えています。** 用語集の説明文は 60〜210 文字（JSON に包むと 96 トークンを超える項目が多い）で、96 のままでは学習後の出力が途中で切れて `json_error` になり、学習の効果が見えなくなるためです。生成の上限を伸ばしただけで、学習条件は変えていません。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃えます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、adapter と結果ファイルを Drive にも保存します。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（116 件）を何周学習するか。1 周 15 ステップなので合計 300 ステップ（A100 で 5〜10 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,    # 元は 96。用語集の説明文が JSON 込みで 96 トークンを超えるため、切れないように伸ばしている（唯一の変更点）
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "train_file": "/content/glossary_sft.jsonl",                       # 学習データ（無ければセクション 2 でアップロードを求める）
    "work_dir": "/content/qlora_lab_glossary",                         # 結果ファイルの保存先
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab_glossary",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 192,
  "gen": {
    "do_sample": false
  },
  "train_file": "/content/glossary_sft.jsonl",
  "work_dir": "/content/qlora_lab_glossary",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab_glossary"
}


## 2. 用語集の学習データと評価データ

**やること**: `glossary_sft.jsonl` を読み込んで学習用（train）を作り、評価用（test）の 12 問と正解 `gold` を定義します。

ファイルはリポジトリの `data/glossary_sft.jsonl`（`scripts/make_sft_dataset.py` の出力）に置き、セルの先頭の `TRAIN_URL`（raw URL）から取得します。Colab で GitHub から開いた場合リポジトリのファイルは `/content` に無いためです。fork の名前やブランチが違う場合は URL を直してください。取得に失敗したときはアップロードを求めます。

モデルへの入力は「質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です（元の演習と同じ形式）。

- 用語集にある質問には、用語集の説明文で答え、`needs_review: false` にする。学習データの assistant の文をそのまま `answer` に入れます。
- 用語集に無い質問には、「用語集に掲載がありません。正確な説明には確認が必要です。」と答え、`needs_review: true` にする。**この例は学習データにはありません。**

**データの分け方**

- train は `glossary_sft.jsonl` の 116 件すべて（根拠あり 116、根拠なし 0）です。1 語につき「〜とは何ですか？」「〜について簡単に説明してください。」の 2 件があります。元の演習と違い、検証用（valid）は分けません。
- test は 12 問（根拠あり 8、根拠なし 4）で、3 種類に分かれます。A: 用語集にある語を質問に含める（4 問。「既定モデル」「answer.md」など）。B: 用語名を出さずに言い回しを変えて聞く（4 問。「どの環境変数を設定しますか」など）。C: 用語集にない語（4 問）。A・B の答えはいずれも講座固有の固有名詞（`gemini-3.5-flash-lite`、`AIE_MODEL`、`lec01`、`requires_approval` など）で、学習前のモデルが当てられるものではありません。どの質問も学習データの文面とは一致しません。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。語句は固有名詞なので、含まれていれば用語集の内容を使ったと見なせます。

**見るところ**: `train=116 (58 語) test=12 gold=12` と、学習例の 1 件。評価の質問が学習データと重複していると、ここでエラーになります。

In [4]:
import re

# 学習データはリポジトリ（自分の fork）の data/glossary_sft.jsonl から取る。fork 名やブランチが違う場合はここを直す
TRAIN_URL = "https://raw.githubusercontent.com/nagayoshi3/lecture-ai-engineering-2026/main/data/glossary_sft.jsonl"

train_path = Path(CONFIG["train_file"])
if not train_path.exists():
    # 1) raw URL から取得する。2) 取れなければアップロードを求める
    import urllib.request
    try:
        with urllib.request.urlopen(TRAIN_URL, timeout=30) as r:
            train_path.write_bytes(r.read())
        print("取得:", TRAIN_URL)
    except Exception as e:
        print(f"URL から取得できませんでした（{type(e).__name__}）。glossary_sft.jsonl をアップロードしてください")
        from google.colab import files
        uploaded = files.upload()
        train_path.write_bytes(uploaded[next(iter(uploaded))])

UNGROUNDED_ANSWER = "用語集に掲載がありません。正確な説明には確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "target": target, "needs_review": needs_review}

def split_msgs(rec):
    user = next(m["content"] for m in rec["messages"] if m["role"] == "user")
    asst = next(m["content"] for m in rec["messages"] if m["role"] == "assistant")
    return user, asst

# 学習データ: 用語集の説明文を answer に入れ、needs_review=false にする（根拠なしの例は無い）
train = [make_example(q, a, False) for q, a in
         (split_msgs(json.loads(l)) for l in train_path.read_text(encoding="utf-8").splitlines() if l.strip())]
TERMS = sorted({m.group(1) for r in train if (m := re.match(r"「(.+?)」とは何ですか", r["question"]))})

# 評価データ: (質問, 参考の模範解答, 正解の needs_review, 正解の answer に含まれるべき語)
TEST_SPEC = [
    # A. 用語集にある語を質問に含める（答えは講座固有の事実。事前学習の一般知識では答えられない）
    ("AIE2026の既定モデルは何ですか？", "gemini-3.5-flash-lite です。無料枠のレート制限と講座運営の安定性を考えて選ばれています。", False, ["gemini-3.5-flash-lite"]),
    ("answer.md はどこにあり、何を書くファイルですか？", "lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを書きます。", False, ["lectures/NN/answer.md"]),
    ("宿題の3レベルとは、どの3つですか？", "基礎（講義内容の確認）、実装（自分でコードを書く）、発展（調査や試行）の3段階です。", False, ["基礎", "実装", "発展"]),
    ("Pydantic AIでシステムプロンプトを与えるのは、Agent のどの引数ですか？", "instructions 引数です。", False, ["instructions"]),
    # B. 言い回しを変えた質問（用語名を出さずに、用語集の中の事実を聞く）
    ("演習で使うモデルを別のものに切り替えたいとき、どの環境変数を設定しますか？", "環境変数 AIE_MODEL です。", False, ["AIE_MODEL"]),
    ("本講座では、回ごとの作業用にどのような名前のブランチを作りますか？", "lec01 のような、回ごとの作業ブランチを作ります。", False, ["lec01"]),
    ("前の回が終わっていなくてもその回の演習を始められるように、講座repoのコードに付けられているものは何ですか？", "各回の演習開始時点のコードに付けられた lecNN-start のような Git タグです。", False, ["タグ"]),
    ("Pydantic AIで、実行前に人の承認を求めるツールにするには何を指定しますか？", "requires_approval=True を指定します。", False, ["requires_approval"]),
    # C. 用語集にない語（needs_review=true が正解）
    ("ベクトルデータベースとは何ですか？", UNGROUNDED_ANSWER, True, ["確認"]),
    ("プロンプトインジェクションとは何ですか？", UNGROUNDED_ANSWER, True, ["確認"]),
    ("知識蒸留とは何ですか？", UNGROUNDED_ANSWER, True, ["確認"]),
    ("AIE2026の修了証はいつ発行されますか？", UNGROUNDED_ANSWER, True, ["確認"]),
]
test, gold = [], {}
for q, ref, nr, must in TEST_SPEC:
    test.append(make_example(q, ref, nr)); gold[q] = {"needs_review": nr, "must_include": must}

all_questions = [r["question"] for r in train + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / test のどこかで重複しています"
print(f"train={len(train)} ({len(TERMS)} 語) test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=116 (58 語) test=12 gold=12
学習例の 1 件 | 質問: 「AIエンジニアリング」とは何ですか？ | 望ましい出力: {"answer": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。", "needs_review": false}


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 12 問に答えさせて採点します。

プロンプト（`SYSTEM`）には元と同じく「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。**形式は指示済み** です。元との違いは、FAQ の代わりになる用語集をプロンプトに含めないことだけです（user の内容は「質問: 〜」のみ）。

**採点のしかた**（元と同じ関数。セクション 5 でも使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「固定」）が `answer` に含まれること。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01`（重みは 32bit で保持）。モデルのダウンロードに 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、12 問の出力の表。学習前は、多くの出力で `needs_review` が欠けている（`missing_key`）はずです。内容は、A・B の講座固有の事実を知らないので、別のモデル名や環境変数名を作って答える（内容 ×）か「確認が必要」と逃げる（これも正解は `false` なので内容 ×）かのどちらかで、根拠なしの語（ベクトルデータベースなど）は一般知識で説明して断定するはずです。`content_rate` は 0 に近い値が予想されます。

**出ても問題ない警告**: `You are sending unauthenticated requests to the HF Hub`、`... is falling back to its reference PyTorch implementation ...` は元の演習と同じで、無視して構いません。

出力の原文は `eval_before.json` にも保存されています。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点（元の演習の FAQ を用語集に置き換えただけ）
SYSTEM = "あなたは講座の用語集の回答係です。用語集だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。用語集に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    """JSON として読め、必須キーと型が合うか。失敗理由も返す"""
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    """内容の合否: needs_review が正解と一致し、必須語句を answer に含む"""
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    """test の 12 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する"""
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は用語集と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 12, "format_rate": 0.08333333333333333, "content_rate": 0.0, "both_rate": 0.0, "seconds": 45.6}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,AIE2026の既定モデルは何ですか？,False,"{""answer"": ""AIE2026 の既定モデルは、AIE2025 のモデルである「AIE2025」です。""}",×,missing_key,×
1,answer.md はどこにあり、何を書くファイルですか？,False,"{""answer"": ""answer.md は通常、回答の作成者（または作成元）のファイル名として設定されています。回答係として確認が必要かどうかは、回答の作成元が「回答係」であるか、あるいは「回答係」として設定されているかによって異なりますが、一般的な用語集の回答係として設定されている場合、answer.md は「確認が必要」となります。""}",×,missing_key,×
2,宿題の3レベルとは、どの3つですか？,False,"{""answer"": ""宿題の3レベルとは、レベル1（基礎）、レベル2（応用）、レベル3（実践）の3つです。""}",×,missing_key,×
3,Pydantic AIでシステムプロンプトを与えるのは、Agent のどの引数ですか？,False,"{""answer"": ""system_prompt""}",×,missing_key,×
4,演習で使うモデルを別のものに切り替えたいとき、どの環境変数を設定しますか？,False,"{""answer"": ""パラメータ""}",×,missing_key,×
5,本講座では、回ごとの作業用にどのような名前のブランチを作りますか？,False,"{""answer"": ""master""}",×,missing_key,×
6,前の回が終わっていなくてもその回の演習を始められるように、講座repoのコードに付けられているものは何ですか？,False,"{""answer"": ""講座repoのコードに付けられているものは、その回の演習を開始するためのリソース（例：演習ファイル、演習コード、演習設定ファイルなど）です。""}",×,missing_key,×
7,Pydantic AIで、実行前に人の承認を求めるツールにするには何を指定しますか？,False,"{""answer"": ""pydantic_ai"", ""needs_review"": false}",○,,×
8,ベクトルデータベースとは何ですか？,True,"{""answer"": ""ベクトルデータベースは、データがベクトル（数値列）の形式で保存・操作されるデータベースです。ベクトルは、長さの異なる数値列であり、ベクトルデータベースはベクトルを基本単位として扱うため、ベクトル操作（加算、乗算、投影など）が基本動作となります。""}",×,missing_key,×
9,プロンプトインジェクションとは何ですか？,True,"{""answer"": ""プロンプトインジェクションとは、攻撃者が攻撃対象のシステムやアプリケーションに、攻撃者の意図した特定の文脈や情報を入力して、そのシステムに不正な動作や攻撃を行う手法です。""}",×,missing_key,×


## 4. LoRA 学習（A100 で 5〜10 分）

LoRA と QLoRA の違いは元のノートブックのセクション 4 を参照してください。この演習でもモデルは量子化なしで T4 に載るので、LoRA で学習します。

### 学習の実行

**やること**: train の 116 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（300 ステップ）学習します。設定（`target_modules`、バッチ 4 × 勾配累積 2、学習率、cosine、warmup）は元と同じです。A100 向けに `USE_AMP = True`（計算だけ bf16 の混合精度。重みは 32bit のまま）にしています。学習した adapter は `adapter/` に保存します。

- 1 エポックは 15 ステップ（8 件 × 14 と端数 4 件）で、20 エポックで 300 ステップです。各例をちょうど 20 回学習します。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || trainable%: 0.84` 前後。
- `Training Loss` の表（10 ステップごと）: 元と同様に 1.5 前後から始まり、終盤で 0.01〜0.05 まで下がる見込みです。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 300 | エポック: 20.0`。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...`。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = True   # A100 向け: 計算だけ bf16 の混合精度にする（重みは 32bit のまま）。T4 は bf16 非対応なので False にする（fp16 は nan になることがあるため使わない）
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                bf16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
steps_per_epoch = -(-len(train) // 8)
print(f"1 エポック {steps_per_epoch} ステップ × {CONFIG['num_train_epochs']} エポック = {steps_per_epoch*CONFIG['num_train_epochs']} ステップの予定")
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421
1 エポック 15 ステップ × 20 エポック = 300 ステップの予定


Step,Training Loss
10,2.409951
20,1.119592
30,0.855814
40,0.516885
50,0.297529
60,0.183914
70,0.093994
80,0.079190
90,0.068499
100,0.059472


学習時間: 1331 秒 | ステップ数: 300 | エポック: 20.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから続きを学習します。

**A100 が確保できず T4 で動かす場合**: 学習セルの `USE_AMP` を `False` にしてください（T4 は bf16 非対応）。学習は 20〜30 分かかります。時間内に終わらない場合は `CONFIG["num_train_epochs"]` を 5 に下げると 75 ステップ（6〜8 分）で終わりますが、元の演習と設定が違うことをセクション 7 の提出物に明記してください。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 12 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、用語集の別項目を当てはめて断定した。学習データに根拠なしの例が無いので、この型が残りやすいはずです。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。
  - B の言い回しを変えた質問に、別の用語の説明文を返した（必須語句が無く内容 ×）。学習は「用語名 → 説明」の向きだけなので、用語名を出さない聞き方は A より難しいはずです。A と B の正解数を分けて見てください。

学習後の回答は、教師データの文（用語集の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「用語集に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても GPU 上の学習は完全には再現しません。どの質問が残るかは実行ごとに変わることがあるので、失敗の型で比べてください。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 12, "format_rate": 1.0, "content_rate": 0.25, "both_rate": 0.25, "seconds": 48.0}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,12,0.083333,0.00,0.00,45.6
after,12,1.000000,0.25,0.25,48.0


退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,AIE2026の既定モデルは何ですか？,False,"{""answer"": ""AIE2026 の既定モデルは、AIE2025 のモデルである「AIE2025」です。""}",×/×,"{""answer"": ""AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。"", ""needs_review"": false}",○/○
1,answer.md はどこにあり、何を書くファイルですか？,False,"{""answer"": ""answer.md は通常、回答の作成者（または作成元）のファイル名として設定されています。回答係として確認が必要かどうかは、回答の作成元が「回答係」であるか、あるいは「回答係」として設定されているかによって異なりますが、一般的な用語集の回答係として設定されている場合、answer.md は「確認が必要」となります。""}",×/×,"{""answer"": ""AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。"", ""needs_review"": false}",○/○
2,宿題の3レベルとは、どの3つですか？,False,"{""answer"": ""宿題の3レベルとは、レベル1（基礎）、レベル2（応用）、レベル3（実践）の3つです。""}",×/×,"{""answer"": ""AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。"", ""needs_review"": false}",○/○
3,Pydantic AIでシステムプロンプトを与えるのは、Agent のどの引数ですか？,False,"{""answer"": ""system_prompt""}",×/×,"{""answer"": ""モデルに渡す指示する文章。質問の内容や、ツールの名前・説明・引数の意味をモデルが理解して、その上でツールを呼び出す。Pydantic AIの文档では「システムプロンプト」と呼ぶが、Claude DesktopやCursorなどのエディタでは「ツールオプション」と扱う。"", ""needs_review"": false}",○/×
4,演習で使うモデルを別のものに切り替えたいとき、どの環境変数を設定しますか？,False,"{""answer"": ""パラメータ""}",×/×,"{""answer"": ""環境変数 GEMINI_PROTOTYPE で、現在のモデルを複製してテストモデルとして使う。テスト中に実際に使うのは、設定した実際のモデル。"", ""needs_review"": false}",○/×
5,本講座では、回ごとの作業用にどのような名前のブランチを作りますか？,False,"{""answer"": ""master""}",×/×,"{""answer"": ""AIE2026の回ごとに区別する命名規約のブランチ（main の左側に付ける）。例：pyd623-aie26-xx-x。"", ""needs_review"": false}",○/×
6,前の回が終わっていなくてもその回の演習を始められるように、講座repoのコードに付けられているものは何ですか？,False,"{""answer"": ""講座repoのコードに付けられているものは、その回の演習を開始するためのリソース（例：演習ファイル、演習コード、演習設定ファイルなど）です。""}",×/×,"{""answer"": ""前の回が終わっていなくても、この回の演習を始められるように設定している（MCPの「ツール」として公開している）。ツールは「呼び出し文」で説明し、モデルはそれを読んで演習を始めよう。"", ""needs_review"": false}",○/×
7,Pydantic AIで、実行前に人の承認を求めるツールにするには何を指定しますか？,False,"{""answer"": ""pydantic_ai"", ""needs_review"": false}",○/×,"{""answer"": ""Pydantic AIでツールが実行される先を指定する設定値。指定したモデルは、このモデルのDocstringで使う承認メッセージ。モデルがメッセージを読み込んだときに、この設定値からモデル名を抽出して、対応するAPIを呼び出す。"", ""needs_review"": false}",○/×
8,ベクトルデータベースとは何ですか？,True,"{""answer"": ""ベクトルデータベースは、データがベクトル（数値列）の形式で保存・操作されるデータベースです。ベクトルは、長さの異なる数値列であり、ベクトルデータベースはベクトルを基本単位として扱うため、ベクトル操作（加算、乗算、投影など）が基本動作となります。""}",×/×,"{""answer"": ""入力と出力の型をベクトルで指定したツール。Gemini APIの vector_database_tool_use setting にある値が正しいかを確認する。"", ""needs_review"": false}",○/×
9,プロンプトインジェクションとは何ですか？,True,"{""answer"": ""プロンプトインジェクションとは、攻撃者が攻撃対象のシステムやアプリケーションに、攻撃者の意図した特定の文脈や情報を入力して、そのシステムに不正な動作や攻撃を行う手法です。""}",×/×,"{""answer"": ""モデルが入力のプロンプト中に隠した攻撃データやツール呼び出しの経路を認識し、その中に自分のアカウントで呼び出せるように設定したシステムプロンプト。モデルは既に知られているので、一度学習したまま、切り口（cut）を切って使う。"", ""needs_review"": false}",○/×


## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 元の演習と同じく、用語集にも学習データにも無い事実を 3 問尋ねます（対照として用語集にある質問を 1 問、学習データの文面そのままで加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答の分類（`format_error` / `needs_review` / `asserted`）は元と同じです。

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。断定した場合に何を根拠にしたように見えるか（用語集の別項目を流用しているか）を読んでください。対照の 1 問は、学習データの文がどのくらいそのまま再現されるかの確認にもなります。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [9]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # 用語集に無い
    "担当教員の名前は何ですか",                     # 用語集に無い
    "次回の講義のテーマは何ですか",                 # 用語集に無い
    "「LoRA」とは何ですか？",                       # 用語集にある（対照。学習データの文面そのもの）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

train_qs = {r["question"] for r in train}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "用語集にある": q in train_qs, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                           "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,用語集にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,この講座の最終課題の締切はいつですか,False,asserted,"{""answer"": ""2024 年 10 月 15 日"", ""needs_review"": false}",asserted,"{""answer"": ""解答式で出力するHugging Face Hubのrepo。テストデータと模範解答が入っている。"", ""needs_review"": false}"
1,担当教員の名前は何ですか,False,format_error,"{""answer"": ""担当教員の名前は「山田 太郎」です。""}",asserted,"{""answer"": ""質問に絞った会話で決める。AIE2026では各回の演習で習う担当教員がいる。演習が複数あるときは、担当教員が講義の回数を数えて呼び出せる。"", ""needs_review"": false}"
2,次回の講義のテーマは何ですか,False,format_error,"{""answer"": ""次回の講義のテーマは「AI 倫理と社会への影響」です""}",asserted,"{""answer"": ""AIエンジニアリングの最新情報をまとめ、演習と試験の準備を支援する講座の回答係。Pydantic AIとCopilot Engineを使えるようにプログラムを渡す。"", ""needs_review"": false}"
3,「LoRA」とは何ですか？,True,format_error,"{""answer"": ""低秩自回帰（Low-Rank Adaptation）""}",asserted,"{""answer"": ""事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。"", ""needs_review"": false}"


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab_glossary` にあります）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果
- 本人が学習を実行した範囲。エポック数を変えた場合はその旨。

In [10]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir", "train_file")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし、bf16 混合精度）" if USE_AMP else "LoRA（量子化なし）",
    "train_size": len(train), "n_terms": len(TERMS),
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし、bf16 混合精度）",
  "train_size": 116,
  "n_terms": 58,
  "before": {
    "label": "before",
    "n": 12,
    "format_rate": 0.08333333333333333,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 45.6
  },
  "after": {
    "label": "after",
    "n": 12,
    "format_rate": 1.0,
    "content_rate": 0.25,
    "both_rate": 0.25,
    "seconds": 48.0
  },
  "regressed": [],
  "executed_training_myself": true
}


## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

元の演習と同じです。同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。学習はしないので、32bit の学習前（`before`）との差が量子化の影響です。

In [11]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.bfloat16,            # 計算精度: bf16（A100 向け。T4 では torch.float16 にする）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

重みのメモリ(GB) | 量子化なし（32bit）: 3.01 | 4bit: 0.76
{"label": "before_4bit", "n": 12, "format_rate": 0.25, "content_rate": 0.0, "both_rate": 0.0, "seconds": 36.4}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,12,0.083333,0.0,0.0,45.6
before_4bit,12,0.250000,0.0,0.0,36.4


,質問,正解の needs_review,4bit・学習前の出力,形式/内容,形式の失敗理由
0,AIE2026の既定モデルは何ですか？,False,"{""answer"": ""AIE2026 の既定モデルは、AIE2026 の「AIE2026」のバージョン名です。""}",×/×,missing_key
1,answer.md はどこにあり、何を書くファイルですか？,False,"{""answer"": ""answer.md は、回答係が回答するテキストファイルです。""}",×/×,missing_key
2,宿題の3レベルとは、どの3つですか？,False,"{""answer"": ""宿題の3レベルとは、レベル1、レベル2、レベル3です""}",×/×,missing_key
3,Pydantic AIでシステムプロンプトを与えるのは、Agent のどの引数ですか？,False,"{""answer"": ""system_prompt"", ""needs_review"": false}",○/×,
4,演習で使うモデルを別のものに切り替えたいとき、どの環境変数を設定しますか？,False,"{""answer"": ""環境変数：\""model_type\""""}",×/×,missing_key
5,本講座では、回ごとの作業用にどのような名前のブランチを作りますか？,False,"{""answer"": ""master""}",×/×,missing_key
6,前の回が終わっていなくてもその回の演習を始められるように、講座repoのコードに付けられているものは何ですか？,False,"{""answer"": ""講座repoのコードに付けられているものは、`README.md` 内の「[本番環境設定](https://github.com/your-org/your-repo/blob/main/README.md)」に記載されている環境設定です。""}",×/×,missing_key
7,Pydantic AIで、実行前に人の承認を求めるツールにするには何を指定しますか？,False,"{""answer"": ""pydantic_ai"", ""needs_review"": false}",○/×,
8,ベクトルデータベースとは何ですか？,True,"{""answer"": ""ベクトルデータベースは、ベクトル（座標値を含む数値）を管理するデータベースです。ベクトルは、数値の組み合わせとして表現され、その組み合わせをベクトルとして表現します。ベクトルデータベースは、ベクトルを管理するデータベースとして機能します。""}",×/×,missing_key
9,プロンプトインジェクションとは何ですか？,True,"{""answer"": ""プロンプトインジェクションとは、AI 助手がユーザーの意図を無視して、特定の目的や文脈に特化した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や、ユーザーの意図を無視して生成した、不自然な回答や",×/×,json_error: JSONDecodeError
